In [ ]:
import time
import os
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt


DATA_DIR = Path("../data/training/fully_periodic_small_hex")

pairwise_dis_ce = [
    DATA_DIR / "PairDis_Infhex_1_CC.npy"
]

pairwise_dis_ed_cen = [
    DATA_DIR / "PairDis_Infhex_1_EC.npy"
]

Actual_data = [
    DATA_DIR / "dspacing_PMF.out"
]




device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

def load_pairwise_distances(file_path):
    data = np.load(file_path, allow_pickle=True).item()  # Load as dictionary
    if not isinstance(data, dict):
        raise ValueError(f"Loaded data is not a dictionary: {type(data)} from {file_path}")


    sorted_keys = sorted(data.keys())
    d_spacing = torch.tensor(sorted_keys, dtype=torch.float32, device=device)
    distances = [torch.tensor(data[key], dtype=torch.float32, device=device) for key in sorted_keys]

    return d_spacing, distances

# Load actual energies and pairwise distances
processed_data = {}
actual_energies = {}

for i, file_path in enumerate(pairwise_dis_ce):
    dataset_key = f'dataset_{i+1}'
    processed_data[dataset_key] = {}

    # Load pairwise distances
    d_spacing_cc, dist_cc = load_pairwise_distances(file_path)
    processed_data[dataset_key]["d_spacing"] = d_spacing_cc
    processed_data[dataset_key]["center_center"] = dist_cc  # List of tensors
    

    print(f"✔ Loaded {file_path} - {len(dist_cc)} d-spacing values.")
    
for i, file_path in enumerate(pairwise_dis_ed_cen):
    dataset_key = f'dataset_{i+1}'
#     processed_data[dataset_key] = {}

    # Load pairwise distances
    d_spacing_ec, dist_ec = load_pairwise_distances(file_path)
    processed_data[dataset_key]["d_spacing"] = d_spacing_ec
    processed_data[dataset_key]["edge_center"] = dist_ec  # List of tensors
    

    print(f"✔ Loaded {file_path} - {len(dist_ec)} d-spacing values.")

# Load actual energy data
for i, file_path in enumerate(Actual_data):
    dataset_key = f'dataset_{i+1}'
    data = np.loadtxt(file_path, delimiter=' ')

    # Sort data by d-spacing
    sorted_indices = np.argsort(data[:, 0])
    data = data[sorted_indices]  # Apply sorting

    d_space = torch.tensor(data[:, 0], dtype=torch.float32, device=device)
    energy_values = torch.tensor(data[:, 1], dtype=torch.float32, device=device)

    # Store in dictionaries
    processed_data[dataset_key]["d_spacing_energy"] = d_space
    processed_data[dataset_key]["actual"] = energy_values
    actual_energies[dataset_key] = energy_values

    # Check if d-spacing values match between .npy and .out files
    if not torch.allclose(d_space, processed_data[dataset_key]["d_spacing"]):
        print(f"⚠ Warning: d-spacing mismatch between .npy and .out files for {dataset_key}")

    print(f"✔ Loaded {file_path} - {len(data)} rows sorted.")

# Verify loaded data
for key in processed_data:
    print(f"\nDataset: {key}")
    print(f"  d_spacing (from .npy): {processed_data[key]['d_spacing'].shape}")
    print(f"  d_spacing (from .out): {processed_data[key]['d_spacing_energy'].shape}")
    print(f"  center_center distances: {len(processed_data[key]['center_center'])} arrays")
    print(f"  Actual energies: {processed_data[key]['actual'].shape}")

    # Print first 5 d-spacing values and corresponding distances/energies
    print("\nFirst 5 d-spacing values and corresponding data:")
    for i in range(5):
        print(f"  d-spacing: {processed_data[key]['d_spacing'][i].item()}")
        print(f"    Number of distances: {len(processed_data[key]['center_center'][i])}")
        print(f"    Energy: {processed_data[key]['actual'][i].item()}")


In [ ]:
#Optimized CC from previous stages
morse_cc = torch.tensor([-200.3445,    0.8255,   -2.2312], device=device)
num_S = 38
S_points = torch.tensor([0.0000,  1.0417,  2.0833,  2.6000,  3.1250,  3.6000,  4.1667,  4.6000,
         5.2083,  6.2500,  6.6000,  7.2917,  7.6000,  8.3333,  8.4000,  9.3750,
         9.6000, 10.4000, 10.4167, 11.4583, 11.6000, 12.4000, 12.5000, 13.5417,
        13.6000, 14.5833, 14.6000, 15.6000, 15.6250, 16.6667, 17.7083, 18.7500,
        19.7917, 20.8333, 21.8750, 22.9167, 23.9583, 25.0000
], device=device)


l_cc = torch.tensor(0.7842, device=device)

coeffs_cc = torch.tensor([0.0000e+00,  0.0000e+00,  6.3404e-07,  1.8891e+00,  8.1625e+00,
         1.0581e-01, -2.4098e+00, -4.0319e-01,  7.9287e-01, -1.9698e-01,
        -6.0995e-02,  7.0925e-02, -4.9648e-02, -5.2572e-02,  4.9121e-02,
         5.2969e-02, -2.1126e-02,  2.1917e-02, -5.2105e-02,  3.6958e-02,
        -4.6758e-02,  5.6710e-02, -1.1014e-02, -3.1777e-03, -1.9909e-02,
         6.7422e-03,  1.3268e-02, -1.1562e-03, -6.4095e-03,  4.7646e-03,
        -3.3277e-03,  2.0708e-03, -1.1522e-03,  6.7129e-04, -4.4138e-04,
         3.0538e-04, -1.4848e-04,  3.1906e-05
], device=device)

morse_ec = torch.tensor([-60.6008,   0.8054,  -0.4834], device=device)


def morse_potential(r, D_e, a, r_e):
    delta = a * (r - r_e)
    exp_term = torch.exp(-delta)
    return D_e * (1 - exp_term)**2 - D_e

def calculate_kernels(distances, S_points, l):
    l = torch.clamp(l, min=1e-3)  # Ensure l > 0

    # Compute squared differences for all pairs (distances, S_points)
    diff = (distances.unsqueeze(1) - S_points) ** 2 / (2 * (l ** 2))  # Ensure 3D shape

    # Compute Gaussian kernel
    kernel_values = torch.exp(-diff)

    # Compute cutoff value at r = 3l
    cutoff_value = torch.exp(-((3 * l) ** 2) / (2 * (l ** 2)))

    # Apply cutoff: subtract cutoff_value
    kernel_values = kernel_values.clone() - cutoff_value  

    # Apply masking for cutoff
    mask = torch.abs(distances.unsqueeze(1) - S_points) > 3 * l
    kernel_values = torch.where(mask, torch.tensor(0.0, device=l.device), kernel_values)

    return kernel_values  # Ensures (N, S) shape


def precompute_cc_contributions(dist_cc, morse_cc, S_points, l_cc, coeffs_cc):
    morse_cc_list = []
    pred_cc_list = []

    for dist_tensor in dist_cc:
        # Morse potential energy
        morse_energy = morse_potential(dist_tensor, *morse_cc).sum()
        morse_cc_list.append(morse_energy)

        # GPR contribution
        K_cc = calculate_kernels(dist_tensor, S_points, l_cc)
        pred_cc = torch.einsum('ns,s->n', K_cc, coeffs_cc).sum()
        pred_cc_list.append(pred_cc)

    # Convert lists to tensors
    morse_cc_tensor = torch.stack(morse_cc_list)
    pred_cc_tensor = torch.stack(pred_cc_list)

    return morse_cc_tensor, pred_cc_tensor

def precompute_ec_morse(dist_ec, morse_ec):
    morse_ec_list = []

    for dist_tensor in dist_ec:

        morse_energy = morse_potential(dist_tensor, *morse_ec).sum()
        morse_ec_list.append(morse_energy)


    morse_ec_tensor = torch.stack(morse_ec_list)

    return morse_ec_tensor


precomputed = {}
for dkey in processed_data:
    dist_cc = processed_data[dkey]["center_center"]
    dist_ec = processed_data[dkey]["edge_center"]
    
    morse_cc_tensor, pred_cc_tensor = precompute_cc_contributions(dist_cc, morse_cc, S_points, l_cc, coeffs_cc)
    morse_ec_tensor = precompute_ec_morse(dist_ec, morse_ec)
    
    precomputed[dkey] = {
        "morse_cc": morse_cc_tensor,
        "pred_cc": pred_cc_tensor,
        "morse_ec": morse_ec_tensor
    }
        


        



In [ ]:
class GPR_EC_Model(nn.Module):
    def __init__(self, num_S=38):
        super().__init__()
        
        self.num_S = num_S
        self.S_points = torch.tensor([0.0000,  1.0417,  2.0833,  2.6000,  3.1250,  3.6000,  4.1667,  4.6000,
         5.2083,  6.2500,  6.6000,  7.2917,  7.6000,  8.3333,  8.4000,  9.3750,
         9.6000, 10.4000, 10.4167, 11.4583, 11.6000, 12.4000, 12.5000, 13.5417,
        13.6000, 14.5833, 14.6000, 15.6000, 15.6250, 16.6667, 17.7083, 18.7500,
        19.7917, 20.8333, 21.8750, 22.9167, 23.9583, 25.0000
        ], device=device)
        

        self.l_ec = nn.Parameter(torch.tensor(1.0, device=device))
        self.coeffs_ec = nn.Parameter(torch.zeros(38, device=device))  # 38 zeros
        self.critical_d = 22
    
    
    def smooth_cutoff(self, d):
        """Smooth transition to zero between cutoff_start and cutoff_end"""
        return torch.sigmoid((self.cutoff_end - d) * 10.0)  # 10 controls steepness
    
    
    def calculate_kernels(self, distances, S_points, l):
        l = torch.clamp(l, min=1e-3)  # Ensure l > 0

        # Compute squared differences for all pairs (distances, S_points)
        diff = (distances.unsqueeze(1) - S_points) ** 2 / (2 * (l ** 2))  # Ensure 3D shape

        # Compute Gaussian kernel
        kernel_values = torch.exp(-diff)

        # Compute cutoff value at r = 3l
        cutoff_value = torch.exp(-((3 * l) ** 2) / (2 * (l ** 2)))

        # Apply cutoff: subtract cutoff_value
        kernel_values = kernel_values.clone() - cutoff_value  

        # Apply masking for cutoff
        mask = torch.abs(distances.unsqueeze(1) - S_points) > 3 * l
        kernel_values = torch.where(mask, torch.tensor(0.0, device=l.device), kernel_values)

        return kernel_values  # Ensures (N, P, S) shape
    
    def forward(self, data):
        total_loss = 0.0
        for dkey in data.keys():
            dist_ec = data[dkey]["edge_center"]

            actual = data[dkey]["actual"]
            d_spacing = data[dkey]["d_spacing"]
            
            morse_cc_tensor = precomputed[dkey]["morse_cc"]
            morse_ec_tensor = precomputed[dkey]["morse_ec"]
            pred_cc_tensor  = precomputed[dkey]["pred_cc"]
    
            # Kernel predictions
            pred_ec_list = []
            for dist_tensor in dist_ec:
                K_ec = self.calculate_kernels(dist_tensor, self.S_points, self.l_ec)
                pred_ec = torch.einsum('ns,s->n', K_ec, self.coeffs_ec).sum()
                pred_ec_list.append(pred_ec)
                
            pred_ec_tensor = torch.stack(pred_ec_list)
            predicted = morse_cc_tensor + pred_cc_tensor + morse_ec_tensor + pred_ec_tensor
            
            
            w = 1 / (torch.abs(actual) + 1e-3)
            w = torch.clamp(w, min=1/90, max=1)  # Prevent extreme imbalance


            mse_loss = torch.mean(w * (predicted - actual)**2)
            total_loss += mse_loss
#             reg_loss = 0.01 * torch.norm(self.coeffs_cc, p=2) + 0.001 * torch.norm(self.l_cc, p=2)
                        
            
#             left_region = d_spacing < self.critical_d
#             right_region = ~left_region
            
#             # Focused left-region loss
#             left_weights = 1.0 / (0.5 + torch.abs(actual[left_region]))
#             left_loss = torch.mean(left_weights * (predicted[left_region] - actual[left_region])**2)
            
#             # Right-region penalty (unchanged)
#             zero_penalty = 30.0 * torch.mean(torch.abs(predicted[right_region]))
            
#             # Selective regularization
#             reg_loss = 0.01 * torch.norm(self.coeffs_ec[10:], p=1)
            
#             total_loss += left_loss + zero_penalty + reg_loss
            
        return total_loss


model = GPR_EC_Model().to(device)
optimizer = optim.LBFGS(
    model.parameters(),
    lr=0.5,               # Start with moderate learning rate
    max_iter=20,          # Inner iterations
    history_size=100,
    line_search_fn='strong_wolfe'  # More stable
)

# Training with LBFGS
loss_history = []
best_loss = float('inf')
patience = 50
no_improve = 0

def closure():
    optimizer.zero_grad()
    loss = model(processed_data)
    loss.backward()
    
    # Optional: Apply gradient constraints
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    
    # Apply parameter constraints
    with torch.no_grad():
        model.l_ec.data.clamp_(0.5, 2.5)
    
    return loss

for epoch in range(1, 501):  # Fewer epochs needed for LBFGS
    loss = optimizer.step(closure)
    loss = loss.item()
    loss_history.append(loss)
    
    # Early stopping
    if loss < best_loss - 1e-5:
        best_loss = loss
        no_improve = 0
        best_state = model.state_dict()
    else:
        no_improve += 1
    
    if no_improve >= patience:
        print(f"Early stopping at epoch {epoch}")
        model.load_state_dict(best_state)
        break
    
#     if epoch % 10 == 0:
    print(f"Epoch {epoch}: Loss={loss:.6f}, l_ec={model.l_ec.item():.4f}")
    print(f"Coef norm: {torch.norm(model.coeffs_ec).item():.4f}")

# Visualize results
plt.figure(figsize=(12,4))
plt.subplot(131)
plt.semilogy(loss_history)
plt.title("Loss History")

plt.subplot(132)
plt.bar(range(model.num_S), model.coeffs_ec.detach().cpu().numpy())
plt.title("Final Coefficients")

plt.tight_layout()
plt.show()

print("\nOptimization complete!")
print(f"Final loss: {best_loss:.6f}")
print(f"Length scale: {model.l_ec.item():.6f}")    



In [ ]:
# Final EC GPR parameters used in the model
l_ec = torch.tensor(0.9172, device=device)

coeffs_ec = torch.tensor([1.0000e-02,  1.0000e-02,  2.7176e-01,  1.8848e+00,  2.6622e+00,
        -8.0390e-01,  2.4433e+00,  5.5400e-02, -3.3196e+00,  2.7279e+00,
         4.3200e-02, -1.8347e+00, -2.9200e-02,  1.0011e+00,  2.5900e-02,
        -4.3774e-01, -5.2200e-02,  3.0100e-02,  1.9532e-01, -1.3795e-01,
        -6.3000e-03, -1.1700e-02,  1.2568e-01, -1.1154e-01, -7.1000e-03,
         9.7802e-02,  0.0000e+00,  0.0000e+00, -7.8401e-02,  6.1313e-02,
        -4.4903e-02,  3.1545e-02, -2.1325e-02,  1.4045e-02, -8.1299e-03,
         3.8430e-03, -9.9988e-04, -2.5886e-04
], device=device)



def predict_energy(data):
    """Predict energy using optimized parameters."""
    predicted_energies = {}

    for dkey in data.keys():

        dist_ec = data[dkey]["edge_center"]
        actual = data[dkey]["actual"]  # Tensor of actual energies

        morse_cc_tensor = precomputed[dkey]["morse_cc"]
        morse_ec_tensor = precomputed[dkey]["morse_ec"]
        pred_cc_tensor  = precomputed[dkey]["pred_cc"]
    
            # Kernel predictions
        pred_ec_list = []
        for dist_tensor in dist_ec:
            K_ec = calculate_kernels(dist_tensor, S_points, l_ec)
            pred_ec = torch.einsum('ns,s->n', K_ec, coeffs_ec).sum()
            pred_ec_list.append(pred_ec)
                
        pred_ec_tensor = torch.stack(pred_ec_list)
        predicted = morse_cc_tensor + pred_cc_tensor + morse_ec_tensor + pred_ec_tensor
        predicted_energies[dkey] = predicted

    return predicted_energies


def plot_results(dkey, processed_data, predicted_energies, zoom=False):
    actual = processed_data[dkey]["actual"].cpu().numpy()
    predicted = predicted_energies[dkey].cpu().numpy()
    xvals = processed_data[dkey]["d_spacing_energy"].cpu().numpy()
    
    plt.figure(figsize=(8, 6))
    plt.plot(xvals, actual, 'o-', label="Atomistic", alpha=0.7)
    plt.plot(xvals, predicted, 's-', label="Mesoscale", markersize=5)
    
    plt.xlabel("d-spacing (Å)", fontsize=12)
    plt.ylabel("PMF (kcal/mol)", fontsize=12)
    plt.title(f"Predicted vs Actual Energies: {dkey}", fontsize=14)
    plt.legend(fontsize=12)
    
    if zoom:
        plt.ylim([-2, 5])
        plt.xlim([9, 30])
        plt.xticks(np.arange(9, 32, 1))
    else:
        plt.xticks(np.arange(9, 30, 2))
    
    ax = plt.gca()
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    plt.grid(True, alpha=0.3)
    plt.tight_layout()

predicted_energies = predict_energy(processed_data)

for dkey in processed_data.keys():
    plot_results(dkey, processed_data, predicted_energies)

